# Test du baoulé avec omniASR-CTC-1B (Meta) sur Kaggle

Ce notebook me sert à écouter comment `facebook/omniASR-CTC-1B` transcrit un extrait audio en baoulé, et à comparer le résultat à une transcription humaine quand j'en ai une sous la main.

**Avant de lancer quoi que ce soit, dans les paramètres du notebook :**
1. Activer **Internet** (pour télécharger le modèle).
2. Activer un **accélérateur GPU**.
3. Ajouter le fichier audio via **Add Input**, de préférence en WAV ou FLAC.

**Ensuite :**
1. Lancer l'installation, puis redémarrer le kernel si des bibliothèques ont été remplacées.
2. Renseigner `AUDIO_PATH` avec le chemin du fichier et choisir un extrait contenant une phrase complète.

Quelques précisions techniques : le modèle s'appelle `omniASR_CTC_1B` dans la bibliothèque Meta (c'est bien le même que `facebook/omniASR-CTC-1B` sur le Hub), et le paramètre `lang` n'a aucun effet sur les modèles CTC — pas besoin de s'en soucier. Ce test ne présage pas de la qualité générale du modèle en baoulé : l'idée est d'écouter l'extrait et de juger le résultat au cas par cas.

Sources : [fiche du modèle](https://huggingface.co/facebook/omniASR-CTC-1B), [pipeline officiel](https://github.com/facebookresearch/omnilingual-asr/blob/main/src/omnilingual_asr/models/inference/pipeline.py).


In [ ]:
# Installation officielle + lecture audio et évaluation.
%pip install -q omnilingual-asr soundfile scipy jiwer


## Vérification de l'environnement

`fairseq2` embarque des composants compilés (C++/CUDA). Si l'import échoue avec une erreur mentionnant Torch, CUDA ou un symbole manquant, ce n'est pas un problème de transcription mais un souci de compatibilité d'environnement (versions de Torch/CUDA/fairseq2 qui ne correspondent pas entre elles).


In [ ]:
import sys
from importlib.metadata import version
import torch

print("Python :", sys.version)
for package in ["omnilingual-asr", "fairseq2", "torch", "torchaudio"]:
    print(package, version(package))
print("CUDA PyTorch :", torch.version.cuda)
if not torch.cuda.is_available():
    raise RuntimeError("Active un GPU dans les paramètres Kaggle, puis relance cette cellule.")
print("GPU :", torch.cuda.get_device_name(0))
# FP32 sur les anciens GPU pour ce premier test ; BF16 si pris en charge.
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32
print("Précision :", DTYPE)


In [ ]:
from pathlib import Path

extensions = {".wav", ".flac", ".ogg", ".mp3", ".m4a"}
files = sorted(p for p in Path("/kaggle/input").rglob("*") if p.suffix.lower() in extensions)
for p in files[:100]:
    print(p)
if not files:
    print("Ajoute ton fichier audio au notebook via Add Input.")


In [ ]:
AUDIO_PATH = "/kaggle/input/datasets/jaureslayone/test-dat/audio.wav"
START_SECONDS = 0.0
DURATION_SECONDS = 20.0  # phrase complète, au maximum 40 secondes.


In [ ]:
import math
import numpy as np
import soundfile as sf
from scipy.signal import resample_poly
from IPython.display import Audio, display

path = Path(AUDIO_PATH)
if not path.is_file():
    raise FileNotFoundError(f"Corrige AUDIO_PATH : {path}")
if START_SECONDS < 0 or not 0 < DURATION_SECONDS <= 40:
    raise ValueError("Début >= 0 et durée comprise entre 0 (exclu) et 40 secondes.")
# Lire seulement l'extrait choisi, même si le fichier original est long.
with sf.SoundFile(str(path)) as f:
    sr = f.samplerate
    start = round(START_SECONDS * sr)
    if start >= len(f):
        raise ValueError("Le début choisi est après la fin de l'audio.")
    f.seek(start)
    samples = f.read(round(DURATION_SECONDS * sr), dtype="float32", always_2d=True)

waveform = samples.mean(axis=1)  # Mono
if sr != 16000:
    divisor = math.gcd(sr, 16000)
    waveform = resample_poly(waveform, 16000 // divisor, sr // divisor)
waveform = np.ascontiguousarray(waveform, dtype=np.float32)
if len(waveform) < 1600 or not np.isfinite(waveform).all():
    raise ValueError("Extrait trop court (< 0,1 s) ou échantillons invalides.")
print(f"Extrait analysé : {START_SECONDS:.2f} à {START_SECONDS + len(waveform)/16000:.2f} s")
print(f"Durée : {len(waveform)/16000:.2f} s ; mono ; 16 kHz")
display(Audio(waveform, rate=16000))
# Si SoundFile ne lit pas ton format (notamment M4A), convertis-le en WAV/FLAC.


In [ ]:
from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline

# Premier lancement : téléchargement des poids (plusieurs Go).
pipeline = ASRInferencePipeline(
    model_card="omniASR_CTC_1B",
    device="cuda",
    dtype=DTYPE,
)


In [ ]:
import time

torch.cuda.synchronize()
t0 = time.perf_counter()
transcription = pipeline.transcribe(
    [{"waveform": waveform, "sample_rate": 16000}],
    batch_size=1,
)[0]
torch.cuda.synchronize()
print("TRANSCRIPTION :")
print(transcription)
print(f"Temps d'inférence et prétraitement : {time.perf_counter() - t0:.2f} s")

output_dir = Path("/kaggle/working")
output_dir.mkdir(parents=True, exist_ok=True)
(output_dir / "transcription_baoule.txt").write_text(transcription, encoding="utf-8")
sf.write(str(output_dir / "extrait_baoule.wav"), waveform, 16000)
print("Texte et extrait enregistrés dans /kaggle/working.")


## Évaluation de la qualité (facultatif)

Pour mesurer objectivement la qualité de la transcription, il faut d'abord écrire la transcription humaine de **l'extrait écouté** (j'utilise le dataset waxal-bau—tts) idéalement avant de lire le résultat du modèle, histoire de ne pas être influencé. Autant garder une orthographe cohérente d'un extrait à l'autre.

Deux métriques standards permettent la comparaison :
- **WER** (taux d'erreur par mot) : substitutions + suppressions + insertions, divisées par le nombre de mots de la référence.
- **CER** (taux d'erreur par caractère) : même principe, au niveau des caractères.

Plus le score est bas, mieux c'est ; 0 % correspond à une correspondance parfaite. Le WER peut dépasser 100 % s'il y a beaucoup d'insertions. Un seul extrait donne un premier aperçu, pas une conclusion valable pour tous les locuteurs du baoulé.

La normalisation ci-dessous harmonise seulement l'encodage Unicode, la casse et les espaces. Accents, tons et ponctuation restent tels quels, donc ils comptent comme des différences si la référence et la sortie du modèle ne les écrivent pas de la même façon.


In [ ]:
REFERENCE = """"""  # Insère ici la transcription humaine exacte de l'extrait.

import unicodedata
from jiwer import wer, cer

def normalize(text):
    return " ".join(unicodedata.normalize("NFC", text).lower().split())

reference = normalize(REFERENCE)
hypothesis = normalize(transcription)
if reference:
    print(f"WER : {100 * wer(reference, hypothesis):.2f} %")
    print(f"CER : {100 * cer(reference, hypothesis):.2f} %")
else:
    print("Renseigne REFERENCE pour calculer WER et CER.")


## Pour la suite

Pour tester un autre passage, il suffit de modifier `START_SECONDS` et `DURATION_SECONDS`, puis de relancer la lecture, la transcription et l'évaluation

En cas de mémoire GPU insuffisante, réduire d'abord la durée de l'extrait.

Le pipeline CTC n'accepte pas plus de 40 secondes par extrait. Pour un premier test fiable, mieux vaut choisir une phrase entière de 10 à 20 secondes, sans couper un mot en plein milieu.
